# 연구 결과 보고와 재현성

## 분석결과를 표·그림·주장·한계로 연결하기

앞 교시의 결과를 처음부터 다시 실행하고, 같은 증거를 가리키는 표·그림·결과문장·한계와 재현성 기록을 완성합니다.


## 이번 실습에서 완성할 것

- Bike 결과경로는 같은 설정으로 재실행해 저장 지표와 일치하는지 확인합니다.
- 개인·조별 계획경로는 계획을 결과처럼 꾸미지 않고 사용권·정보시점·다음 실행을 보고합니다.
- 두 경로 모두 한계·환경·artifact hash와 실제 3분 발표문을 남깁니다.

**완료 결과:** Bike는 표·그림·결과·한계문장, 개인·조별은 결과수치 없는 계획 보고서; 공통으로 3분 발표와 재현성 기록


## 이번 교시에서 사용할 데이터

### 강의 시연: Bike 예측결과 보고 예시

UCI Bike Sharing의 2011~2012년 시간별 자료를 사용합니다. 한 행은 날짜×시간 관측이고, `cnt`는 해당 시간의 총 대여건수입니다. 시간순 train/test 경계, feature, baseline, 후보모형과 MAE를 다시 명시한 뒤 표와 그림으로 보고합니다.

### 적용 실습: 11교시 개인·조별 분석결과

새 데이터나 새 모형을 추가하지 않고 `artifacts/lesson11_project_record.json`을 읽어 앞 교시의 경로와 상태를 그대로 이어갑니다.

| 11교시 기록 | 12교시 보고 원칙 |
|---|---|
| `bike / analysis_result` | 같은 variant·feature·분할로 다시 실행하고 저장된 지표와 일치하는지 확인 |
| `personal_canvas / analysis_plan` | 계획·사용권·정보시점·다음 실행을 보고하고 모형·성능수치·결과주장을 새로 만들지 않음 |

파일을 찾지 못하면 11교시 Practice를 먼저 완료합니다. 다음 항목이 모두 같은 분석을 가리키는지 확인합니다.

- 데이터 출처·snapshot 또는 접근일
- 한 행의 의미와 최종 분석 표본
- target·feature·제외 열
- train/test 또는 비교집단 정의
- baseline·후보방법·metric·random seed
- 표·그림·결과문장·한계문장

Bike 예시는 형식을 보여주기 위한 자료입니다. 개인·조별 경로가 아직 `analysis_plan`이면 계획을 결과처럼 바꾸지 않고, 다음 실행과 검증질문을 발표합니다.


## 11교시 기록을 먼저 확인하기

| artifact의 경로·상태 | 그대로 유지할 항목 | 이번 교시에서 완성할 항목 |
|---|---|---|
| `bike / analysis_result` | variant·feature·시간경계·baseline·후보·seed | 지표 재현, 표·그림·결과·한계문장 |
| `personal_canvas / analysis_plan` | 사용권·관측 단위·정보시점·비교계획 | 결과수치 없는 계획 보고, 다음 실행·검증질문 |

`artifacts/lesson11_project_record.json`을 찾지 못하면 11교시 Practice를 먼저 완료합니다. 이번 교시에서 바꾸는 것은 모형이나 계획의 상태가 아니라 **보고의 완결성**입니다.


## 실제 실행 순서와 산출물

1. 새 kernel에서 첫 셀부터 실행하고 L11 artifact의 경로·hash·`track`·`status`를 확인합니다.
2. Bike 경로는 같은 설정으로 다시 실행해 저장된 두 지표와 일치하는지 확인합니다.
3. 개인·조별 경로는 계획·사용권·정보시점·다음 실행을 옮기고 결과수치를 만들지 않습니다.
4. 현재 경로의 근거·주장 또는 계획상태·한계가 같은 기록을 가리키는지 확인합니다.
5. 실제로 말할 내용이 들어간 3분 발표표와 재현성 기록을 완성합니다.

**산출물:** Bike는 결과표·그림·결과·한계문장, 개인·조별은 성능수치 없는 계획 보고서; 공통으로 artifact hash·발표문 5칸·재현성 기록


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 새 실행환경에서 분석 재현

발표 자료가 이전 kernel 상태에 의존하지 않도록 데이터·feature·split·model을 다시 명시합니다.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import hashlib
import json
import sklearn

artifact_candidates = [
    Path.cwd() / "artifacts" / "lesson11_project_record.json",
    *[parent / "artifacts" / "lesson11_project_record.json" for parent in Path.cwd().parents],
]
artifact_path = next((path for path in artifact_candidates if path.exists()), None)
if artifact_path is None:
    raise FileNotFoundError("11교시 Practice를 먼저 완료해 artifacts/lesson11_project_record.json을 만드세요.")

project_record = json.loads(artifact_path.read_text(encoding="utf-8"))
REPORT_PATH = project_record.get("track")
artifact_bytes = artifact_path.read_bytes()
artifact_sha256 = hashlib.sha256(artifact_bytes).hexdigest()
artifact_size_bytes = len(artifact_bytes)

if REPORT_PATH == "bike":
    if project_record.get("status") != "analysis_result":
        raise ValueError("Bike artifact의 status가 analysis_result가 아닙니다.")
    BIKE_VARIANT = project_record["bike_variant"]
    feature_options = {
        "calendar_and_lag": ["lag_1h", "hr", "weekday", "workingday"],
        "observed_weather_benchmark": [
            "lag_1h", "hr", "weekday", "workingday", "weathersit", "temp", "hum", "windspeed"
        ],
    }
    if BIKE_VARIANT not in feature_options:
        raise ValueError(f"지원하지 않는 Bike variant: {BIKE_VARIANT}")
    features = project_record["features"]
    if features != feature_options[BIKE_VARIANT]:
        raise ValueError("L11 artifact의 variant와 feature 목록이 일치하지 않습니다.")

    data_path = find_course_data(project_record["data_relative_path"])
    data_bytes = data_path.read_bytes()
    data_sha256 = hashlib.sha256(data_bytes).hexdigest()
    data_size_bytes = len(data_bytes)
    raw = pd.read_csv(data_path, parse_dates=["dteday"])
    raw["timestamp"] = raw["dteday"] + pd.to_timedelta(raw["hr"], unit="h")
    raw = raw.sort_values("timestamp")
    lag = raw[["timestamp", "cnt"]].copy(); lag["timestamp"] += pd.Timedelta(hours=1)
    data = raw.merge(lag.rename(columns={"cnt": "lag_1h"}), on="timestamp", how="left").dropna(subset=["lag_1h"]).copy()
    cut = int(len(data) * 0.80); train, test = data.iloc[:cut], data.iloc[cut:]
    MODEL_SPEC = "HistGradientBoostingRegressor(max_iter=120, learning_rate=0.07, random_state=42)"
    model = HistGradientBoostingRegressor(max_iter=120, learning_rate=0.07, random_state=42).fit(train[features], train["cnt"])
    prediction = model.predict(test[features]); baseline = test["lag_1h"].to_numpy()
    TARGET_TIME_WEATHER_USED = project_record.get("target_time_observed_weather", [])
    WEATHER_BENCHMARK_NOTE = (
        "이 결과는 결과시각의 실제 날씨 관측값을 사용할 수 있다고 가정한 조건부 benchmark입니다. 실제 운영 예측에는 예보값과 예보오차 검증이 필요합니다."
        if TARGET_TIME_WEATHER_USED else
        "결과시각의 실제 날씨 관측값을 사용하지 않은 calendar-and-lag 분석입니다."
    )
    execution_scope = pd.DataFrame({
        "항목": ["L11 artifact", "L11 track·variant", "데이터 snapshot", "train 마지막 시각", "test 첫 시각", "입력변수"],
        "기록": [str(artifact_path), f"{REPORT_PATH} · {BIKE_VARIANT}", f"{data_path.name} | {data_size_bytes} bytes | sha256 {data_sha256}",
                 str(train["timestamp"].max()), str(test["timestamp"].min()), features],
    })
    display(execution_scope)
else:
    if REPORT_PATH != "personal_canvas" or project_record.get("status") != "analysis_plan":
        raise ValueError("L11 artifact의 track/status 조합을 확인하세요.")
    BIKE_VARIANT = None
    features = []
    data_path = data = train = test = model = prediction = baseline = None
    data_sha256 = None; data_size_bytes = None
    planning_canvas = pd.DataFrame(project_record["canvas"])
    print("L11 개인·조별 분석계획을 불러왔습니다. 결과수치가 없는 계획을 결과보고로 바꾸지 않습니다.")
    display(planning_canvas)

print("L11 artifact sha256:", artifact_sha256)
print("L11 artifact size:", artifact_size_bytes, "bytes")


## 2. 연구질문에 직접 답하는 근거의 선택

결과표는 baseline과 후보를 같은 단위로 비교하고, 그림 caption은 표본·기간·단위를 포함합니다.


In [ ]:
if REPORT_PATH == "bike":
    result_table = pd.DataFrame([
        {"model":"previous-hour persistence", "MAE_rentals_per_hour":mean_absolute_error(test["cnt"], baseline), "RMSE_rentals_per_hour":root_mean_squared_error(test["cnt"], baseline)},
        {"model":"hist gradient boosting", "MAE_rentals_per_hour":mean_absolute_error(test["cnt"], prediction), "RMSE_rentals_per_hour":root_mean_squared_error(test["cnt"], prediction)},
    ])
    display(result_table)
    result_by_model = result_table.set_index("model")
    candidate_mae = float(result_by_model.loc["hist gradient boosting", "MAE_rentals_per_hour"])
    baseline_mae = float(result_by_model.loc["previous-hour persistence", "MAE_rentals_per_hour"])
    mae_difference = baseline_mae - candidate_mae
    evaluation_period = f"{test['timestamp'].min():%Y-%m-%d %H:%M}–{test['timestamp'].max():%Y-%m-%d %H:%M}"

    l11_metrics = pd.DataFrame(project_record["metrics"]).set_index("model")
    l11_candidate_mae = float(l11_metrics.loc["hist gradient boosting", "MAE"])
    l11_baseline_mae = float(l11_metrics.loc["persistence", "MAE"])
    metric_handoff_matches = np.isclose(candidate_mae, l11_candidate_mae) and np.isclose(baseline_mae, l11_baseline_mae)

    import matplotlib.pyplot as plt
    view = 168
    fig, ax = plt.subplots(figsize=(10.5, 4.3))
    ax.plot(test["timestamp"].iloc[:view], test["cnt"].iloc[:view], label="observed")
    ax.plot(test["timestamp"].iloc[:view], baseline[:view], label="persistence baseline", alpha=0.85)
    ax.plot(test["timestamp"].iloc[:view], prediction[:view], label="candidate")
    figure_start = test["timestamp"].iloc[0]; figure_end = test["timestamp"].iloc[view - 1]
    ax.set(title=f"Figure 1. {BIKE_VARIANT} | {figure_start:%Y-%m-%d}–{figure_end:%Y-%m-%d}", xlabel="Timestamp", ylabel="Bike rentals/hour")
    ax.legend(); plt.xticks(rotation=25); plt.tight_layout(); plt.show()
    print(f"표는 전체 평가기간 {evaluation_period}, 그림은 그중 첫 168시간을 보여 줍니다.")
else:
    result_table = result_by_model = None
    candidate_mae = baseline_mae = mae_difference = evaluation_period = None
    metric_handoff_matches = True
    print("분석계획 경로이므로 성능표와 예측그림을 만들지 않습니다.")


## 3. 근거 범위에 맞는 결과문장

숫자보다 강한 주장을 하지 않도록 claim type과 보고 조건을 명시합니다.


In [ ]:
if REPORT_PATH == "bike":
    mae_direction = "낮았습니다" if mae_difference >= 0 else "높았습니다"
    claim = (
        f"{evaluation_period}의 시간순 평가자료에서 HistGradientBoosting의 MAE는 {candidate_mae:.2f}건/시간으로, "
        f"previous-hour persistence의 {baseline_mae:.2f}건/시간보다 {abs(mae_difference):.2f}건/시간 {mae_direction}."
    )
    limitation = project_record["limitation"]
    reproducibility_record = pd.Series({
        "source": "UCI Bike Sharing, DOI 10.24432/C5W894",
        "lesson11_artifact": str(artifact_path),
        "artifact_sha256": artifact_sha256,
        "artifact_size_bytes": artifact_size_bytes,
        "data_snapshot": data_path.name,
        "data_sha256": data_sha256,
        "data_size_bytes": data_size_bytes,
        "bike_variant": BIKE_VARIANT,
        "evaluation_period": evaluation_period,
        "features": features,
        "model": project_record["model"],
        "python": platform.python_version(), "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "claim_type": "prediction performance on held-out future period",
        "limitation": limitation,
    }, name="record")
    claim_view = pd.DataFrame({
        "기록 유형": ["근거", "결과문장", "한계"],
        "내용": [result_table.round(3).to_string(index=False), claim, limitation],
    })
    print("CLAIM:", claim); print("LIMITATION:", limitation)
    display(reproducibility_record.to_frame()); display(claim_view)
else:
    claim = None
    limitation = project_record["limitation"]
    planning_report = pd.DataFrame({
        "보고 항목": ["상태", "자료 출처·사용권", "정보 기준시점", "비교 기준", "평가척도", "다음 실행"],
        "내용": ["분석계획 — 결과 없음", project_record["data_source_and_license"], project_record["information_timing"],
                 project_record["baseline"], project_record["metric"], project_record["next_execution"]],
    })
    reproducibility_record = pd.Series({
        "lesson11_artifact": str(artifact_path),
        "artifact_sha256": artifact_sha256,
        "artifact_size_bytes": artifact_size_bytes,
        "report_status": "analysis plan; no model or metrics",
        "python": platform.python_version(), "pandas": pd.__version__,
        "limitation": limitation,
    }, name="record")
    display(planning_report); display(reproducibility_record.to_frame())


## 4. 3분 발표 점검표

3분 발표에 문제·방법·결과·한계·질문이 모두 있는지 확인합니다.


In [ ]:
if REPORT_PATH == "bike":
    if TARGET_TIME_WEATHER_USED:
        problem_text = (
            "이번 결과는 결과시각의 실제 날씨 관측값이 주어졌다고 가정한 조건부 benchmark입니다. "
            "한 도시의 한 시간을 한 행으로 두고, 이 조건에서 후보모형이 직전 값 기준보다 오차를 줄였는지 묻습니다. 실제 한 시간 선행예측 성능으로 해석하지 않습니다."
        )
    else:
        problem_text = (
            "운영자는 시간대별 자전거 부족을 줄이기 위해 다음 한 시간의 대여량을 미리 알고 싶습니다. "
            "이번 분석은 한 도시의 한 시간을 한 행으로 두고, 예측 전에 확보 가능한 달력·과거수요만으로 후보모형이 직전 값 기준보다 오차를 줄였는지 묻습니다."
        )
    method_text = (
        f"11교시에서 저장한 {BIKE_VARIANT} 설정과 입력변수 {features}를 그대로 불러왔습니다. "
        f"과거자료는 {train['timestamp'].min():%Y-%m-%d}부터 {train['timestamp'].max():%Y-%m-%d}까지이며, "
        "그 뒤의 미래기간을 test로 두었습니다. 두 방법은 같은 행에서 MAE와 RMSE로 비교했고, 데이터와 L11 기록의 SHA-256도 남겼습니다."
    )
    result_text = (
        claim + " 이 차이는 같은 미래기간에서 계산했으며, 표의 전체 기간과 그림의 첫 168시간 범위를 구분했습니다. "
        "또한 다시 계산한 두 MAE가 11교시 artifact와 일치하는지 확인했습니다."
    )
    limitation_text = (
        limitation + " 따라서 이 결과를 다른 도시·계절이나 정책 개입의 효과로 확대하지 않습니다. "
        "입력정보의 실제 확보시점과 운영비용은 별도의 검증이 필요합니다."
    )
    next_check_text = (
        "다음 단계에서는 예측 기준시점을 여러 번 이동해 계절별 안정성을 확인하고, 실제 배포 시점에 확보되는 입력만 사용해 다시 평가하겠습니다. "
        "청중에게는 현재 입력정보 중 사전에 확보하기 가장 어려운 열이 무엇인지 질문하겠습니다."
    )
else:
    canvas_map = dict(zip(planning_canvas["항목"], planning_canvas["기록"]))
    problem_text = f"이번 발표는 결과보고가 아니라 분석계획 공유입니다. 현실 문제는 {canvas_map['현실 문제']}이고 질문은 {canvas_map['분석 질문']}입니다."
    method_text = (
        f"한 행은 {canvas_map['관측 단위']}, 결과변수는 {canvas_map['결과변수']}입니다. "
        f"정보 기준시점은 {project_record['information_timing']}, 비교 기준은 {project_record['baseline']}, 평가척도는 {project_record['metric']}입니다."
    )
    result_text = "아직 원자료를 분석하거나 모형을 실행하지 않았으므로 성능수치와 결과주장은 없습니다. 현재 산출물은 승인 전 Problem Canvas와 재현 가능한 다음 실행계획입니다."
    limitation_text = limitation + " 자료 사용권과 변수 생성시점이 확인되기 전에는 결과 분석으로 진행하지 않습니다."
    next_check_text = f"다음 실행은 {project_record['next_execution']}입니다. 청중에게는 관측 단위와 비교 기준이 연구질문에 맞는지 검토를 요청하겠습니다."

presentation = pd.DataFrame({
    "시간": ["0:00–0:30", "0:30–1:00", "1:00–2:00", "2:00–2:30", "2:30–3:00"],
    "구성요소": ["문제·질문", "자료·방법", "근거 또는 현재 상태", "한계", "다음 검증·질문"],
    "실제로 말할 내용": [problem_text, method_text, result_text, limitation_text, next_check_text],
})
display(presentation)
presentation_characters = int(presentation["실제로 말할 내용"].str.len().sum())

if REPORT_PATH == "bike":
    checks = pd.Series({
        "L11 setting preserved": BIKE_VARIANT == project_record["bike_variant"] and features == project_record["features"],
        "L11 time boundaries preserved": str(train["timestamp"].max()) == project_record["train_end"] and str(test["timestamp"].min()) == project_record["test_start"],
        "L11 evaluation period preserved": evaluation_period == project_record["evaluation_period"],
        "L11 model preserved": MODEL_SPEC == project_record["model"],
        "L11 metrics reproduced": bool(metric_handoff_matches),
        "claim has period, two MAE, difference, unit": evaluation_period in claim and all(f"{v:.2f}" in claim for v in [candidate_mae, baseline_mae, abs(mae_difference)]) and "건/시간" in claim,
        "snapshot has SHA-256 and size": len(data_sha256) == 64 and data_size_bytes > 0 and len(artifact_sha256) == 64 and artifact_size_bytes > 0,
        "five presentation components complete": list(presentation["구성요소"]) == ["문제·질문", "자료·방법", "근거 또는 현재 상태", "한계", "다음 검증·질문"] and presentation["실제로 말할 내용"].str.strip().ne("").all(),
        "presentation has substantive content": presentation_characters >= 550,
    }, name="passed")
else:
    checks = pd.Series({
        "personal artifact remains a plan": project_record["status"] == "analysis_plan",
        "no model, metrics, or result claim": project_record["model"] is None and project_record["metrics"] is None and claim is None,
        "plan fields carried forward": planning_report["내용"].astype(str).str.strip().ne("").all(),
        "artifact has SHA-256 and size": len(artifact_sha256) == 64 and artifact_size_bytes > 0,
        "five presentation components complete": len(presentation) == 5 and presentation["실제로 말할 내용"].str.strip().ne("").all(),
        "presentation states no result": "성능수치와 결과주장은 없습니다" in result_text,
        "presentation has substantive content": presentation_characters >= 350,
    }, name="passed")

display(checks.to_frame()); assert checks.all()
print("재현성과 보고조건 검증: PASS")

review_payload = result_table.to_string(index=False) if REPORT_PATH == "bike" else planning_report.to_string(index=False)
review_prompt = f"""
다음 L11 연계 기록과 L12 보고안을 검토해 주세요.

- 경로: {REPORT_PATH}
- L11 artifact sha256: {artifact_sha256}
- 보고 내용:
{review_payload}
- 발표문:
{presentation.to_string(index=False)}

1. 판정: 통과 / 수정 필요 / 정보 부족
2. 각 발표문이 같은 L11 기록을 가리키는지
3. 결과보다 강한 표현 또는 계획을 결과처럼 쓴 표현
4. 사람이 다시 확인할 artifact·표·수치

personal_canvas 경로에는 새 성능수치나 결과주장을 만들지 마세요.
""".strip()
print("\n[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


## 제출할 보고 기록

1. **표의 평가기간과 단위:**
2. **그림에 표시한 실제 기간과 전체 표의 기간 차이:**
3. **두 MAE와 차이를 포함한 결과문장:**
4. **결과시각 실측 날씨를 사용한 조건부 benchmark 한계:**
5. **다른 사람이 재실행하는 데 필요한 데이터 snapshot·환경·seed:**
6. **생성형 AI가 도운 범위와 사람이 직접 확인한 수치:**

Code Cell에 출력된 다섯 칸의 발표문을 실제로 소리 내어 읽고, 3분을 넘으면 코드 설명보다 질문·근거·한계·다음 검증을 남깁니다.


## 완료 확인

마지막 출력이 `재현성과 보고조건 검증: PASS`인지 확인합니다. 이 검사는 다음 항목의 형식과 수치 일치를 확인합니다.

- 결과문장에 평가기간, 두 MAE, 차이와 `건/시간` 단위가 있음
- 데이터 snapshot과 실행환경이 기록됨
- 결과시각의 실제 날씨 관측을 쓴 조건부 benchmark 한계가 있음
- 3분 발표의 다섯 구간에 실제로 말할 내용이 있음


## 최종 제출 전 확인

- 결과문장의 모든 숫자를 최종 표 또는 그림에서 다시 찾을 수 있습니까?
- 표본·기간·단위·비교조건과 현재 검증하지 못한 범위가 함께 기록됐습니까?
- 새 kernel의 전체 재실행, 데이터·환경·설정 기록과 생성형 AI 사용 범위 공개를 마쳤습니까?
